In [1]:
import pandas as pd

dataframe = pd.read_csv(
    "../data/sentences.csv.zip",
    index_col=False,
    compression="zip",
)
dataframe.head()

,phrase,target
0,"According to Gran , the company has no plans t...",neutral
1,"For the last quarter of 2010 , Componenta 's n...",positive
2,"In the third quarter of 2010 , net sales incre...",positive
3,Operating profit rose to EUR 13.1 mn from EUR ...,positive
4,"Operating profit totalled EUR 21.1 mn , up fro...",positive


In [2]:
dataframe.shape

(2264, 2)

In [3]:
# Busqueda de valores nulos

dataframe.isna().sum()

phrase    0
target    0
dtype: int64

In [4]:
# Distribucion de las clases
#
# El dataset esta desbalanceado: predecir siempre "neutral" ya da 0.614
# de accuracy, esa es la linea base a superar.

dataframe.target.value_counts(normalize=True)

target
neutral     0.614399
positive    0.251767
negative    0.133834
Name: proportion, dtype: float64

In [5]:
# Longitud de las frases en caracteres

dataframe.phrase.str.len().describe()

count    2264.000000
mean      121.033127
std        56.302653
min         8.000000
25%        78.000000
50%       110.000000
75%       156.000000
max       314.000000
Name: phrase, dtype: float64

In [6]:
# Ejemplos de cada clase

for clase in dataframe.target.unique():
    print(f"--- {clase} ---")
    for frase in dataframe[dataframe.target == clase].phrase.head(2):
        print(frase[:160])
    print()

--- neutral ---
According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing 
At the request of Finnish media company Alma Media 's newspapers , research manager Jari Kaivo-oja at the Finland Futures Research Centre at the Turku School of

--- positive ---
For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m from EUR76m for the same period a year earlier , while it moved to a zero pre-tax prof
In the third quarter of 2010 , net sales increased by 5.2 % to EUR 205.5 mn , and operating profit by 34.9 % to EUR 23.5 mn 

--- negative ---
Jan. 6 -- Ford is struggling in the face of slowing truck and SUV sales and a surfeit of up-to-date , gotta-have cars 
Pharmaceuticals group Orion Corp reported a fall in its third-quarter earnings that were hit by larger expenditures on R&D and marketing 



In [7]:
# Comparacion de vectorizadores y clasificadores por validacion cruzada

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

x, y = dataframe.phrase, dataframe.target
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

vectorizadores = {
    "count_1g": CountVectorizer(stop_words="english"),
    "tfidf_1g": TfidfVectorizer(stop_words="english"),
    "tfidf_12g": TfidfVectorizer(stop_words="english", ngram_range=(1, 2)),
    "tfidf_12g_sin_filtro": TfidfVectorizer(ngram_range=(1, 2)),
}

clasificadores = {
    "nb": MultinomialNB(),
    "logreg": LogisticRegression(max_iter=5000, C=10, random_state=42),
    "svc": LinearSVC(C=1, random_state=42),
}

resultados = []

for nombre_v, vectorizador in vectorizadores.items():
    for nombre_c, clasificador in clasificadores.items():
        modelo = Pipeline(
            [
                ("vectorizer", vectorizador),
                ("classifier", clasificador),
            ]
        )

        scores = cross_val_score(modelo, x, y, cv=cv, scoring="accuracy")
        modelo.fit(x, y)

        resultados.append(
            {
                "modelo": f"{nombre_v} + {nombre_c}",
                "cv_accuracy": scores.mean(),
                "cv_std": scores.std(),
                "accuracy_completo": modelo.score(x, y),
            }
        )

pd.DataFrame(resultados).sort_values("cv_accuracy", ascending=False)

,modelo,cv_accuracy,cv_std,accuracy_completo
11,tfidf_12g_sin_filtro + svc,0.887371,0.012218,1.000000
10,tfidf_12g_sin_filtro + logreg,0.879865,0.009665,1.000000
5,tfidf_1g + svc,0.853355,0.014159,0.992933
8,tfidf_12g + svc,0.851146,0.003697,0.997792
1,count_1g + logreg,0.848061,0.013299,0.998675
2,count_1g + svc,0.845406,0.015102,0.998233
4,tfidf_1g + logreg,0.839666,0.009506,0.995141
7,tfidf_12g + logreg,0.837456,0.004306,0.998233
0,count_1g + nb,0.805213,0.011358,0.918286
9,tfidf_12g_sin_filtro + nb,0.767661,0.012100,0.842756


In [8]:
# Ajuste de hiperparametros del ganador: tfidf sin filtro de stopwords + LinearSVC

resultados = []

for ngram in [(1, 2), (1, 3)]:
    for sublinear in [False, True]:
        for min_df in [1, 2]:
            for c in [0.5, 1, 2, 5]:
                modelo = Pipeline(
                    [
                        (
                            "vectorizer",
                            TfidfVectorizer(
                                ngram_range=ngram,
                                sublinear_tf=sublinear,
                                min_df=min_df,
                            ),
                        ),
                        ("classifier", LinearSVC(C=c, random_state=42)),
                    ]
                )

                scores = cross_val_score(modelo, x, y, cv=cv, scoring="accuracy")

                resultados.append(
                    {
                        "ngram": str(ngram),
                        "sublinear_tf": sublinear,
                        "min_df": min_df,
                        "C": c,
                        "cv_accuracy": scores.mean(),
                        "cv_std": scores.std(),
                    }
                )

pd.DataFrame(resultados).sort_values("cv_accuracy", ascending=False).head(10)

,ngram,sublinear_tf,min_df,C,cv_accuracy,cv_std
11,"(1, 2)",True,1,5.0,0.899740,0.014784
3,"(1, 2)",False,1,5.0,0.898853,0.013335
10,"(1, 2)",True,1,2.0,0.897089,0.012265
14,"(1, 2)",True,2,2.0,0.895763,0.010549
6,"(1, 2)",False,2,2.0,0.893992,0.009481
13,"(1, 2)",True,2,1.0,0.892670,0.011285
5,"(1, 2)",False,2,1.0,0.892668,0.012684
2,"(1, 2)",False,1,2.0,0.892229,0.012101
9,"(1, 2)",True,1,1.0,0.891789,0.010771
30,"(1, 3)",True,2,2.0,0.891788,0.009219


In [9]:
# Entrenamiento de la configuracion elegida sobre todo el dataset

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

x_vectorized = vectorizer.fit_transform(x)

clf = LinearSVC(C=2, random_state=42)
clf.fit(x_vectorized, y)

x_vectorized.shape

(2264, 7249)

In [10]:
# Verificacion de la metrica que evalua el test

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true=y, y_pred=clf.predict(x_vectorized))
accuracy

1.0

In [11]:
# Reporte por clase con validacion cruzada (estimacion honesta)

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

modelo = Pipeline(
    [
        ("vectorizer", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("classifier", LinearSVC(C=2, random_state=42)),
    ]
)

y_pred_cv = cross_val_predict(modelo, x, y, cv=cv)
print(classification_report(y, y_pred_cv))

              precision    recall  f1-score   support

    negative       0.89      0.73      0.80       303
     neutral       0.91      0.97      0.94      1391
    positive       0.87      0.79      0.83       570

    accuracy                           0.90      2264
   macro avg       0.89      0.83      0.86      2264
weighted avg       0.89      0.90      0.89      2264



In [12]:
# Terminos mas asociados a cada clase

import numpy as np

terminos = np.array(vectorizer.get_feature_names_out())

for indice, clase in enumerate(clf.classes_):
    mayores = np.argsort(clf.coef_[indice])[-10:][::-1]
    print(f"{clase:10s}", ", ".join(terminos[mayores]))

negative   down, decreased, fell, decreased to, dropped, down from, lower, declined, fall, lost
neutral    is, mn and, approximately, be, profit eur, the, range, will be, and operating, differ
positive   increased, grew, rose, up, increase, up from, positive, improved, rose to, won


In [13]:
# Serializacion de los artefactos evaluados por el calificador

import pickle

with open("../submission/vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)

with open("../submission/clf.pkl", "wb") as file:
    pickle.dump(clf, file)